In [1]:
import sys
print(sys.executable)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

C:\Users\User\anaconda3\envs\tabpfn_env\python.exe
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 2060


In [2]:
import sys
print(sys.executable)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

C:\Users\User\anaconda3\envs\tabpfn_env\python.exe
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 2060


In [3]:
import math
import torch

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def velocity(x, t):
    """
    Nonlinear time-dependent vector field.

    x: tensor of shape [N, 2]
    t: scalar in [0, 1]
    """
    t = torch.as_tensor(t, dtype=x.dtype, device=x.device)

    x1 = x[:, 0]
    x2 = x[:, 1]

    v1 = (
        0.55 * x1
        + 1.20 * torch.tanh(1.3 * x2)
        + 0.35 * torch.sin(2 * math.pi * t)
    )

    v2 = (
        -0.35 * x2
        + 1.00 * torch.sin(x1)
        + 0.30 * torch.cos(2 * math.pi * t)
    )

    return torch.stack([v1, v2], dim=1)


# sanity check
x_test = torch.randn(5, 2, device=device)
v_test = velocity(x_test, 0.5)

print("x shape:", x_test.shape)
print("v shape:", v_test.shape)
print(v_test)

x shape: torch.Size([5, 2])
v shape: torch.Size([5, 2])
tensor([[ 1.2980, -0.8637],
        [ 0.8676, -0.7684],
        [-0.2298, -1.4667],
        [-1.3013, -0.6186],
        [-0.8255,  0.6501]], device='cuda:0')


In [4]:
def rk4_flow(x0, n_steps=1000):
    """
    Numerically approximates Phi(x0) by integrating
    dx/dt = velocity(x, t) from t=0 to t=1.
    """
    x = x0.clone()
    h = 1.0 / n_steps

    for i in range(n_steps):
        t = i * h

        k1 = velocity(x, t)
        k2 = velocity(x + 0.5 * h * k1, t + 0.5 * h)
        k3 = velocity(x + 0.5 * h * k2, t + 0.5 * h)
        k4 = velocity(x + h * k3, t + h)

        x = x + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return x


N = 5000

z = torch.randn(N, 2, device=device)

with torch.no_grad():
    phi_z = rk4_flow(z, n_steps=1000)

print("z:", z.shape)
print("Phi(z):", phi_z.shape)
print("Example input :", z[0])
print("Example output:", phi_z[0])

z: torch.Size([5000, 2])
Phi(z): torch.Size([5000, 2])
Example input : tensor([ 0.1391, -0.1082], device='cuda:0')
Example output: tensor([0.2690, 0.1229], device='cuda:0')


In [5]:
def euler_flow(x0, n_steps):
    x = x0.clone()
    h = 1.0 / n_steps

    for i in range(n_steps):
        t = i * h
        x = x + h * velocity(x, t)

    return x


budgets = [1, 2, 4, 8, 16, 32]

baselines = {}
residuals = {}
mse_errors = {}

with torch.no_grad():
    for C in budgets:
        B_C = euler_flow(z, n_steps=C)
        R_C = phi_z - B_C

        baselines[C] = B_C
        residuals[C] = R_C

        mse = torch.mean((phi_z - B_C) ** 2).item()
        mse_errors[C] = mse

for C in budgets:
    print(f"C={C:2d} | MSE={mse_errors[C]:.8f}")

C= 1 | MSE=0.26858515
C= 2 | MSE=0.07558003
C= 4 | MSE=0.02134927
C= 8 | MSE=0.00575591
C=16 | MSE=0.00149684
C=32 | MSE=0.00038180


In [6]:
rk_steps = [250, 500, 1000, 2000]
phi_refs = {}

with torch.no_grad():
    for n in rk_steps:
        phi_refs[n] = rk4_flow(z, n_steps=n)

phi_2000 = phi_refs[2000]

for n in rk_steps[:-1]:
    mse = torch.mean((phi_refs[n] - phi_2000) ** 2).item()
    max_err = torch.max(
        torch.linalg.vector_norm(phi_refs[n] - phi_2000, dim=1)
    ).item()

    print(
        f"RK4 steps={n:4d} | "
        f"MSE vs 2000={mse:.12e} | "
        f"max point error={max_err:.12e}"
    )

RK4 steps= 250 | MSE vs 2000=3.396281303067e-12 | max point error=1.689179953246e-05
RK4 steps= 500 | MSE vs 2000=3.652590382930e-12 | max point error=1.860969132395e-05
RK4 steps=1000 | MSE vs 2000=4.459376878180e-12 | max point error=2.005587884923e-05


In [7]:
D = 64
N = 5000

torch.manual_seed(123)

# Dense fixed matrices defining a nonlinear high-dimensional flow
A = torch.randn(D, D, device=device) / (D ** 0.5)
B = torch.randn(D, D, device=device) / (D ** 0.5)

# Keep dynamics numerically stable
A = 0.25 * A
B = 0.60 * B


def velocity_hd(x, t):
    """
    x: [N, D]
    """
    t = torch.as_tensor(t, dtype=x.dtype, device=x.device)

    linear = x @ A.T
    nonlinear = torch.tanh(x @ B.T)

    time_term = 0.15 * torch.sin(2 * math.pi * t) * torch.sin(x)

    return linear + nonlinear + time_term

In [8]:
def rk4_flow_hd(x0, n_steps=500):
    x = x0.clone()
    h = 1.0 / n_steps

    for i in range(n_steps):
        t = i * h

        k1 = velocity_hd(x, t)
        k2 = velocity_hd(x + 0.5*h*k1, t + 0.5*h)
        k3 = velocity_hd(x + 0.5*h*k2, t + 0.5*h)
        k4 = velocity_hd(x + h*k3, t + h)

        x = x + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return x


def euler_flow_hd(x0, n_steps):
    x = x0.clone()
    h = 1.0 / n_steps

    for i in range(n_steps):
        t = i * h
        x = x + h * velocity_hd(x, t)

    return x

In [9]:
z_hd = torch.randn(N, D, device=device)

with torch.no_grad():
    phi_hd = rk4_flow_hd(z_hd, n_steps=500)

print("Input shape :", z_hd.shape)
print("Output shape:", phi_hd.shape)
print("Finite:", torch.isfinite(phi_hd).all().item())
print("Mean abs output:", phi_hd.abs().mean().item())

Input shape : torch.Size([5000, 64])
Output shape: torch.Size([5000, 64])
Finite: True
Mean abs output: 0.9256818890571594


In [10]:
z_check = z_hd[:1000]

with torch.no_grad():
    phi_250  = rk4_flow_hd(z_check, n_steps=250)
    phi_500  = rk4_flow_hd(z_check, n_steps=500)
    phi_1000 = rk4_flow_hd(z_check, n_steps=1000)

for n, phi_n in [(250, phi_250), (500, phi_500)]:
    diff = phi_n - phi_1000

    mse = torch.mean(diff ** 2).item()

    max_err = torch.max(
        torch.linalg.vector_norm(diff, dim=1)
    ).item()

    print(
        f"RK4 steps={n:4d} | "
        f"MSE vs 1000={mse:.12e} | "
        f"max point error={max_err:.12e}"
    )

RK4 steps= 250 | MSE vs 1000=1.046932397546e-12 | max point error=1.478526064602e-05
RK4 steps= 500 | MSE vs 1000=1.247181719878e-12 | max point error=1.602377778909e-05


In [11]:
budgets = [1, 2, 4, 8, 16, 32]

residual_stats = {}

with torch.no_grad():

    for C in budgets:

        # Cheap approximation
        B_C = euler_flow_hd(z_hd, n_steps=C)

        # Residual
        R_C = phi_hd - B_C

        # Baseline approximation error
        mse = torch.mean(R_C ** 2).item()

        # Center residuals across samples
        R_centered = R_C - R_C.mean(dim=0, keepdim=True)

        # Singular values
        s = torch.linalg.svdvals(R_centered)

        # Energy carried by each singular direction
        energy = s**2
        cumulative_energy = torch.cumsum(energy, dim=0) / energy.sum()

        def effective_rank(threshold):
            return int(
                torch.searchsorted(
                    cumulative_energy,
                    torch.tensor(
                        threshold,
                        device=cumulative_energy.device
                    )
                ).item()
            ) + 1

        k90 = effective_rank(0.90)
        k95 = effective_rank(0.95)
        k99 = effective_rank(0.99)

        residual_stats[C] = {
            "mse": mse,
            "k90": k90,
            "k95": k95,
            "k99": k99,
            "singular_values": s.cpu()
        }

        print(
            f"C={C:2d} | "
            f"MSE={mse:.8e} | "
            f"k90={k90:2d} | "
            f"k95={k95:2d} | "
            f"k99={k99:2d}"
        )

C= 1 | MSE=2.05567162e-02 | k90=28 | k95=36 | k99=55
C= 2 | MSE=5.61503740e-03 | k90=28 | k95=37 | k99=55
C= 4 | MSE=1.48771051e-03 | k90=27 | k95=36 | k99=54
C= 8 | MSE=3.86460189e-04 | k90=26 | k95=35 | k99=54
C=16 | MSE=9.86642190e-05 | k90=26 | k95=35 | k99=54
C=32 | MSE=2.49371351e-05 | k90=26 | k95=35 | k99=54


In [12]:
def spectral_complexity(X):
    # Center across samples
    Xc = X - X.mean(dim=0, keepdim=True)

    # Singular values
    s = torch.linalg.svdvals(Xc)

    energy = s**2
    total_energy = energy.sum()

    # Normalized spectral energy
    p = energy / total_energy

    # Entropy effective rank
    entropy = -(p * torch.log(p + 1e-12)).sum()
    entropy_rank = torch.exp(entropy).item()

    # Stable rank
    stable_rank = (energy.sum() / energy.max()).item()

    cumulative = torch.cumsum(p, dim=0)

    def k_for(threshold):
        return int(
            torch.searchsorted(
                cumulative,
                torch.tensor(threshold, device=cumulative.device)
            ).item()
        ) + 1

    return {
        "k90": k_for(0.90),
        "k95": k_for(0.95),
        "k99": k_for(0.99),
        "stable_rank": stable_rank,
        "entropy_rank": entropy_rank
    }

In [13]:
with torch.no_grad():

    full_displacement = phi_hd - z_hd
    full_stats = spectral_complexity(full_displacement)

    print("FULL TRANSPORT DISPLACEMENT")
    print(
        f"k90={full_stats['k90']} | "
        f"k95={full_stats['k95']} | "
        f"k99={full_stats['k99']} | "
        f"stable_rank={full_stats['stable_rank']:.2f} | "
        f"entropy_rank={full_stats['entropy_rank']:.2f}"
    )

    print("\nRESIDUALS")

    for C in budgets:

        B_C = euler_flow_hd(z_hd, n_steps=C)
        R_C = phi_hd - B_C

        stats = spectral_complexity(R_C)

        print(
            f"C={C:2d} | "
            f"k90={stats['k90']:2d} | "
            f"k95={stats['k95']:2d} | "
            f"k99={stats['k99']:2d} | "
            f"stable_rank={stats['stable_rank']:.2f} | "
            f"entropy_rank={stats['entropy_rank']:.2f}"
        )

FULL TRANSPORT DISPLACEMENT
k90=32 | k95=39 | k99=52 | stable_rank=12.10 | entropy_rank=36.45

RESIDUALS
C= 1 | k90=28 | k95=36 | k99=55 | stable_rank=9.88 | entropy_rank=31.00
C= 2 | k90=28 | k95=37 | k99=55 | stable_rank=8.30 | entropy_rank=29.32
C= 4 | k90=27 | k95=36 | k99=54 | stable_rank=7.45 | entropy_rank=27.27
C= 8 | k90=26 | k95=35 | k99=54 | stable_rank=7.08 | entropy_rank=26.34
C=16 | k90=26 | k95=35 | k99=54 | stable_rank=6.90 | entropy_rank=25.88
C=32 | k90=26 | k95=35 | k99=54 | stable_rank=6.82 | entropy_rank=25.65


In [14]:
from sklearn.cluster import KMeans
import numpy as np

# Cluster according to input z-space
z_np = z_hd.detach().cpu().numpy()

K = 10

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(z_np)

print(np.bincount(labels))

[477 487 509 467 502 521 517 511 528 481]


In [15]:
def local_complexity(X, labels):
    local_stats = []

    for cluster_id in np.unique(labels):

        idx_np = np.where(labels == cluster_id)[0]
        idx = torch.tensor(idx_np, device=X.device)

        X_local = X[idx]

        stats = spectral_complexity(X_local)

        stats["n"] = len(idx_np)
        local_stats.append(stats)

    # weighted averages
    total_n = sum(s["n"] for s in local_stats)

    weighted = {}

    for metric in [
        "k90",
        "k95",
        "k99",
        "stable_rank",
        "entropy_rank"
    ]:
        weighted[metric] = sum(
            s[metric] * s["n"] for s in local_stats
        ) / total_n

    return weighted, local_stats

In [16]:
with torch.no_grad():

    print("FULL DISPLACEMENT")

    full_disp = phi_hd - z_hd

    full_local, _ = local_complexity(
        full_disp,
        labels
    )

    print(
        f"k95={full_local['k95']:.2f} | "
        f"stable_rank={full_local['stable_rank']:.2f} | "
        f"entropy_rank={full_local['entropy_rank']:.2f}"
    )

    print("\nRESIDUALS")

    for C in budgets:

        B_C = euler_flow_hd(z_hd, n_steps=C)
        R_C = phi_hd - B_C

        local_stats_C, _ = local_complexity(
            R_C,
            labels
        )

        print(
            f"C={C:2d} | "
            f"k95={local_stats_C['k95']:.2f} | "
            f"stable_rank={local_stats_C['stable_rank']:.2f} | "
            f"entropy_rank={local_stats_C['entropy_rank']:.2f}"
        )

FULL DISPLACEMENT
k95=38.00 | stable_rank=12.27 | entropy_rank=35.17

RESIDUALS
C= 1 | k95=35.81 | stable_rank=9.90 | entropy_rank=30.32
C= 2 | k95=35.90 | stable_rank=8.42 | entropy_rank=28.75
C= 4 | k95=34.81 | stable_rank=7.57 | entropy_rank=26.80
C= 8 | k95=34.31 | stable_rank=7.20 | entropy_rank=25.92
C=16 | k95=34.00 | stable_rank=7.03 | entropy_rank=25.49
C=32 | k95=33.90 | stable_rank=6.94 | entropy_rank=25.28


In [17]:
torch.manual_seed(42)

perm = torch.randperm(N, device=device)

n_train = 4000

train_idx = perm[:n_train]
test_idx  = perm[n_train:]

z_train = z_hd[train_idx]
z_test  = z_hd[test_idx]

residual_train = {}
residual_test = {}

with torch.no_grad():
    for C in budgets:
        B_C = euler_flow_hd(z_hd, n_steps=C)
        R_C = phi_hd - B_C

        residual_train[C] = R_C[train_idx]
        residual_test[C] = R_C[test_idx]

print("Train:", z_train.shape)
print("Test :", z_test.shape)

for C in budgets:
    rms = torch.sqrt(
        torch.mean(residual_train[C] ** 2)
    ).item()

    print(f"C={C:2d} | residual RMS={rms:.6e}")

Train: torch.Size([4000, 64])
Test : torch.Size([1000, 64])
C= 1 | residual RMS=1.433847e-01
C= 2 | residual RMS=7.494482e-02
C= 4 | residual RMS=3.857686e-02
C= 8 | residual RMS=1.966183e-02
C=16 | residual RMS=9.934674e-03
C=32 | residual RMS=4.994571e-03


In [18]:
import torch.nn as nn

class ResidualMLP(nn.Module):
    def __init__(self, d, width):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d, width),
            nn.GELU(),
            nn.Linear(width, width),
            nn.GELU(),
            nn.Linear(width, d)
        )

    def forward(self, x):
        return self.net(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def train_residual_model(
    z_train,
    r_train,
    z_test,
    r_test,
    width,
    epochs=300,
    lr=1e-3,
    batch_size=512,
    seed=123
):
    torch.manual_seed(seed)

    # Normalize target by TRAIN RMS
    rms = torch.sqrt(torch.mean(r_train ** 2))

    y_train = r_train / rms

    model = ResidualMLP(D, width).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    loss_fn = nn.MSELoss()

    n = z_train.shape[0]

    for epoch in range(epochs):

        perm = torch.randperm(n, device=device)

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]

            pred = model(z_train[idx])
            loss = loss_fn(pred, y_train[idx])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():

        # Prediction is in normalized residual units
        pred_normalized = model(z_test)

        # Convert prediction back to original residual scale
        pred_residual = pred_normalized * rms

        # Exact test MSE
        mse = torch.mean(
            (pred_residual - r_test) ** 2
        )

        # Energy of true residual on TEST set
        baseline_energy = torch.mean(
            r_test ** 2
        )

        # Exact normalized MSE
        nmse = (mse / baseline_energy).item()

    return model, nmse

In [19]:
model_test, nmse_test = train_residual_model(
    z_train,
    residual_train[8],
    z_test,
    residual_test[8],
    width=32,
    epochs=300
)

print("Parameters:", count_parameters(model_test))
print("Exact Test NMSE:", nmse_test)

Parameters: 5248
Exact Test NMSE: 0.14341585338115692


In [20]:
widths = [8, 16, 32, 64, 128]

sweep_results = []

for C in budgets:

    print(f"\n===== C = {C} =====")

    for width in widths:

        model, nmse = train_residual_model(
            z_train,
            residual_train[C],
            z_test,
            residual_test[C],
            width=width,
            epochs=300,
            lr=1e-3,
            batch_size=512,
            seed=123
        )

        params = count_parameters(model)

        sweep_results.append({
            "C": C,
            "width": width,
            "params": params,
            "nmse": nmse
        })

        print(
            f"width={width:3d} | "
            f"params={params:7d} | "
            f"NMSE={nmse:.6f}"
        )

        del model


===== C = 1 =====
width=  8 | params=   1168 | NMSE=0.509962
width= 16 | params=   2400 | NMSE=0.298430
width= 32 | params=   5248 | NMSE=0.137895
width= 64 | params=  12480 | NMSE=0.102123
width=128 | params=  33088 | NMSE=0.101750

===== C = 2 =====
width=  8 | params=   1168 | NMSE=0.487754
width= 16 | params=   2400 | NMSE=0.293309
width= 32 | params=   5248 | NMSE=0.148467
width= 64 | params=  12480 | NMSE=0.117198
width=128 | params=  33088 | NMSE=0.116532

===== C = 4 =====
width=  8 | params=   1168 | NMSE=0.463498
width= 16 | params=   2400 | NMSE=0.289358
width= 32 | params=   5248 | NMSE=0.143560
width= 64 | params=  12480 | NMSE=0.113705
width=128 | params=  33088 | NMSE=0.110788

===== C = 8 =====
width=  8 | params=   1168 | NMSE=0.449113
width= 16 | params=   2400 | NMSE=0.278067
width= 32 | params=   5248 | NMSE=0.143416
width= 64 | params=  12480 | NMSE=0.114743
width=128 | params=  33088 | NMSE=0.110032

===== C = 16 =====
width=  8 | params=   1168 | NMSE=0.442658
w

In [21]:
import pandas as pd

results_df = pd.DataFrame(sweep_results)

pivot_nmse = results_df.pivot(
    index="width",
    columns="C",
    values="nmse"
)

print("\nNMSE TABLE")
display(pivot_nmse)


NMSE TABLE


C,1,2,4,8,16,32
width,,,,,,
8,0.509962,0.487754,0.463498,0.449113,0.442658,0.439538
16,0.298430,0.293309,0.289358,0.278067,0.278808,0.277284
32,0.137895,0.148467,0.143560,0.143416,0.143886,0.144284
64,0.102123,0.117198,0.113705,0.114743,0.115536,0.115990
128,0.101750,0.116532,0.110788,0.110032,0.109921,0.109925


In [22]:
thresholds = [0.20, 0.10, 0.05]

for eps in thresholds:

    print(f"\n=== epsilon = {eps} ===")

    for C in budgets:

        subset = (
            results_df[
                (results_df["C"] == C) &
                (results_df["nmse"] <= eps)
            ]
            .sort_values("params")
        )

        if len(subset) == 0:
            print(f"C={C:2d} | not reached")
        else:
            best = subset.iloc[0]

            print(
                f"C={C:2d} | "
                f"min params={int(best['params']):7d} | "
                f"width={int(best['width']):3d} | "
                f"NMSE={best['nmse']:.4f}"
            )


=== epsilon = 0.2 ===
C= 1 | min params=   5248 | width= 32 | NMSE=0.1379
C= 2 | min params=   5248 | width= 32 | NMSE=0.1485
C= 4 | min params=   5248 | width= 32 | NMSE=0.1436
C= 8 | min params=   5248 | width= 32 | NMSE=0.1434
C=16 | min params=   5248 | width= 32 | NMSE=0.1439
C=32 | min params=   5248 | width= 32 | NMSE=0.1443

=== epsilon = 0.1 ===
C= 1 | not reached
C= 2 | not reached
C= 4 | not reached
C= 8 | not reached
C=16 | not reached
C=32 | not reached

=== epsilon = 0.05 ===
C= 1 | not reached
C= 2 | not reached
C= 4 | not reached
C= 8 | not reached
C=16 | not reached
C=32 | not reached


In [23]:
torch.manual_seed(123)

W_linear = torch.randn(D, D, device=device) / (D ** 0.5)
W_nonlinear = torch.randn(D, D, device=device) / (D ** 0.5)

W_linear = 0.25 * W_linear
W_nonlinear = 0.60 * W_nonlinear


def velocity_hd(x, t):
    t = torch.as_tensor(t, dtype=x.dtype, device=x.device)

    linear = x @ W_linear.T
    nonlinear = torch.tanh(x @ W_nonlinear.T)

    time_term = (
        0.15
        * torch.sin(2 * math.pi * t)
        * torch.sin(x)
    )

    return linear + nonlinear + time_term


print(W_linear.shape)
print(W_nonlinear.shape)
print(velocity_hd(z_hd[:10], 0.5).shape)

torch.Size([64, 64])
torch.Size([64, 64])
torch.Size([10, 64])


In [24]:
scaled_residuals = {}

with torch.no_grad():
    for C in budgets:
        B_C = euler_flow_hd(z_hd, n_steps=C)
        R_C = phi_hd - B_C

        scaled_residuals[C] = C * R_C


print("Relative differences between C * R_C:")

for C1, C2 in zip(budgets[:-1], budgets[1:]):

    R1_scaled = scaled_residuals[C1]
    R2_scaled = scaled_residuals[C2]

    relative_diff = (
    torch.mean((R1_scaled - R2_scaled) ** 2)
    / torch.mean(R2_scaled ** 2)
    ).item()
    
    cosine = torch.nn.functional.cosine_similarity(
        R1_scaled.flatten().unsqueeze(0),
        R2_scaled.flatten().unsqueeze(0)
    ).item()

    print(
        f"C={C1:2d} vs C={C2:2d} | "
        f"relative MSE={relative_diff:.6e} | "
        f"cosine={cosine:.6f}"
    )

Relative differences between C * R_C:
C= 1 vs C= 2 | relative MSE=3.237676e-02 | cosine=0.984059
C= 2 vs C= 4 | relative MSE=1.143507e-02 | cosine=0.994536
C= 4 vs C= 8 | relative MSE=2.616913e-03 | cosine=0.998850
C= 8 vs C=16 | relative MSE=6.768674e-04 | cosine=0.999713
C=16 vs C=32 | relative MSE=1.737164e-04 | cosine=0.999928


In [25]:
# Restore the original 64D vector field safely
torch.manual_seed(123)

W_linear = torch.randn(D, D, device=device) / (D ** 0.5)
W_nonlinear = torch.randn(D, D, device=device) / (D ** 0.5)

W_linear = 0.25 * W_linear
W_nonlinear = 0.60 * W_nonlinear


def velocity_hd(x, t):
    t = torch.as_tensor(t, dtype=x.dtype, device=x.device)

    linear = x @ W_linear.T
    nonlinear = torch.tanh(x @ W_nonlinear.T)

    time_term = (
        0.15
        * torch.sin(2 * math.pi * t)
        * torch.sin(x)
    )

    return linear + nonlinear + time_term


print("W_linear:", W_linear.shape)
print("W_nonlinear:", W_nonlinear.shape)
print("velocity test:", velocity_hd(z_hd[:10], 0.5).shape)

W_linear: torch.Size([64, 64])
W_nonlinear: torch.Size([64, 64])
velocity test: torch.Size([10, 64])


In [26]:
def midpoint_flow_hd(x0, n_steps):
    x = x0.clone()
    h = 1.0 / n_steps

    for i in range(n_steps):
        t = i * h

        k1 = velocity_hd(x, t)

        k2 = velocity_hd(
            x + 0.5 * h * k1,
            t + 0.5 * h
        )

        x = x + h * k2

    return x

In [27]:
nfe_budgets = [4, 8, 16, 32]

baseline_comparison = []

with torch.no_grad():

    for C in nfe_budgets:

        # Euler: 1 evaluation per step
        B_euler = euler_flow_hd(
            z_hd,
            n_steps=C
        )

        # RK2: 2 evaluations per step
        B_rk2 = midpoint_flow_hd(
            z_hd,
            n_steps=C // 2
        )

        # RK4: 4 evaluations per step
        B_rk4 = rk4_flow_hd(
            z_hd,
            n_steps=C // 4
        )

        methods = {
            "Euler": B_euler,
            "RK2": B_rk2,
            "RK4": B_rk4
        }

        print(f"\n===== NFE budget C={C} =====")

        for name, B_C in methods.items():

            R_C = phi_hd - B_C

            mse = torch.mean(R_C ** 2).item()

            stats = spectral_complexity(R_C)

            baseline_comparison.append({
                "C": C,
                "method": name,
                "mse": mse,
                "k95": stats["k95"],
                "stable_rank": stats["stable_rank"],
                "entropy_rank": stats["entropy_rank"]
            })

            print(
                f"{name:5s} | "
                f"MSE={mse:.8e} | "
                f"k95={stats['k95']:2d} | "
                f"stable_rank={stats['stable_rank']:.2f} | "
                f"entropy_rank={stats['entropy_rank']:.2f}"
            )


===== NFE budget C=4 =====
Euler | MSE=1.48771051e-03 | k95=36 | stable_rank=7.45 | entropy_rank=27.27
RK2   | MSE=7.19798772e-05 | k95=53 | stable_rank=9.64 | entropy_rank=40.90
RK4   | MSE=6.21187282e-05 | k95=58 | stable_rank=30.09 | entropy_rank=59.56

===== NFE budget C=8 =====
Euler | MSE=3.86460189e-04 | k95=35 | stable_rank=7.08 | entropy_rank=26.34
RK2   | MSE=3.81088921e-06 | k95=49 | stable_rank=7.21 | entropy_rank=31.71
RK4   | MSE=2.67796395e-07 | k95=59 | stable_rank=35.93 | entropy_rank=61.20

===== NFE budget C=16 =====
Euler | MSE=9.86642190e-05 | k95=35 | stable_rank=6.90 | entropy_rank=25.88
RK2   | MSE=2.41459247e-07 | k95=48 | stable_rank=6.77 | entropy_rank=29.88
RK4   | MSE=4.09091983e-10 | k95=58 | stable_rank=22.28 | entropy_rank=58.22

===== NFE budget C=32 =====
Euler | MSE=2.49371351e-05 | k95=35 | stable_rank=6.82 | entropy_rank=25.65
RK2   | MSE=1.52999942e-08 | k95=47 | stable_rank=6.61 | entropy_rank=29.15
RK4   | MSE=1.75845341e-12 | k95=58 | stable_ra

In [28]:
C_test = 8

with torch.no_grad():

    B_euler_8 = euler_flow_hd(
        z_hd,
        n_steps=C_test
    )

    B_rk2_8 = midpoint_flow_hd(
        z_hd,
        n_steps=C_test // 2
    )

    B_rk4_8 = rk4_flow_hd(
        z_hd,
        n_steps=C_test // 4
    )

    residual_methods = {
        "Euler": phi_hd - B_euler_8,
        "RK2":   phi_hd - B_rk2_8,
        "RK4":   phi_hd - B_rk4_8
    }


for name, R in residual_methods.items():

    rms = torch.sqrt(
        torch.mean(R ** 2)
    ).item()

    print(
        f"{name:5s} | "
        f"RMS={rms:.8e}"
    )

Euler | RMS=1.96585916e-02
RK2   | RMS=1.95214991e-03
RK4   | RMS=5.17490495e-04


In [29]:
residual_method_train = {}
residual_method_test = {}

for name, R in residual_methods.items():

    residual_method_train[name] = R[train_idx]
    residual_method_test[name] = R[test_idx]

In [30]:
method_sweep_results = []

widths_test = [8, 16, 32, 64, 128]

for method in ["Euler", "RK2", "RK4"]:

    print(f"\n===== {method} =====")

    for width in widths_test:

        model, nmse = train_residual_model(
            z_train,
            residual_method_train[method],
            z_test,
            residual_method_test[method],
            width=width,
            epochs=300,
            lr=1e-3,
            batch_size=512,
            seed=123
        )

        params = count_parameters(model)

        method_sweep_results.append({
            "method": method,
            "width": width,
            "params": params,
            "nmse": nmse
        })

        print(
            f"width={width:3d} | "
            f"params={params:7d} | "
            f"NMSE={nmse:.6f}"
        )

        del model


===== Euler =====
width=  8 | params=   1168 | NMSE=0.449113
width= 16 | params=   2400 | NMSE=0.278067
width= 32 | params=   5248 | NMSE=0.143416
width= 64 | params=  12480 | NMSE=0.114743
width=128 | params=  33088 | NMSE=0.110032

===== RK2 =====
width=  8 | params=   1168 | NMSE=0.549593
width= 16 | params=   2400 | NMSE=0.409913
width= 32 | params=   5248 | NMSE=0.343534
width= 64 | params=  12480 | NMSE=0.331877
width=128 | params=  33088 | NMSE=0.358101

===== RK4 =====
width=  8 | params=   1168 | NMSE=0.969545
width= 16 | params=   2400 | NMSE=0.956033
width= 32 | params=   5248 | NMSE=0.956364
width= 64 | params=  12480 | NMSE=1.016582
width=128 | params=  33088 | NMSE=1.193814


In [31]:
robustness_results = []

seeds = [11, 22, 33]

for method in ["Euler", "RK2", "RK4"]:

    print(f"\n===== {method} =====")

    for seed in seeds:

        model, nmse = train_residual_model(
            z_train,
            residual_method_train[method],
            z_test,
            residual_method_test[method],
            width=32,
            epochs=500,
            lr=1e-3,
            batch_size=512,
            seed=seed
        )

        robustness_results.append({
            "method": method,
            "seed": seed,
            "nmse": nmse
        })

        print(
            f"seed={seed:2d} | "
            f"NMSE={nmse:.6f}"
        )

        del model


===== Euler =====
seed=11 | NMSE=0.138393
seed=22 | NMSE=0.139308
seed=33 | NMSE=0.141525

===== RK2 =====
seed=11 | NMSE=0.335756
seed=22 | NMSE=0.337517
seed=33 | NMSE=0.338183

===== RK4 =====
seed=11 | NMSE=0.957192
seed=22 | NMSE=0.959320
seed=33 | NMSE=0.959681


In [32]:
robust_df = pd.DataFrame(robustness_results)

summary = (
    robust_df
    .groupby("method")["nmse"]
    .agg(["mean", "std"])
)

display(summary)

,mean,std
method,,
Euler,0.139742,0.001610
RK2,0.337152,0.001254
RK4,0.958731,0.001345
